In [4]:
import pandas as pd

file_path = "Mintt_CLC Live.xlsx"

# Inspect available sheets
excel_file = pd.ExcelFile(file_path)
print("Available sheets in file:", excel_file.sheet_names)

# Auto-detect sheet containing 'july' (case-insensitive) or default to the first sheet
target_sheet = next(
    (s for s in excel_file.sheet_names if "july" in s.lower()), 0
)

df = pd.read_excel(file_path, sheet_name=target_sheet)

print(f"\nSuccessfully loaded sheet: '{target_sheet}'")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
print("First 5 rows:")
df.head()

Available sheets in file: ['August 2026 Leads', 'July 2026 leads ', 'Weeky Ratio', 'May Month', 'april leads-2026', 'Sheet8', 'June Leads', 'CTPRE', 'Sheet4', 'CLC General', 'Dr. Reena Shinde - CLC-13 Nov 2', 'Sheet11', 'CLC - Learning Difficulty - 22 ', 'Hiring CLC - 22 Nov 2025', 'Hiring CLC - 9 Branches - 06 Ja', 'Hiring CLC - Ulwe Nerul Belapur', 'Mintt CLC Form IMP', 'CTPRE Mintt', 'Mintt New Webinar', 'HIRING Mintt']

Successfully loaded sheet: 'July 2026 leads '
Shape: 107 rows, 15 columns

First 5 rows:


,Sr. No,Date,Concerns,Name,Contact number,Email ID,Location,Remark 1,Remark 2,Remark 3,Consultation Done,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14
0,1,2026-07-01,Sonal Yadav,Sonal Yadav,Thane,919834450169,learning_difficulties,Busy on another call 30 June,Name of the child Ananay\nAge :9\nGrade:4\nSho...,She was asking for doctor profile and some tim...,Call Decline 09 July,Call decline 10 July,Call decline 11 July,Call decline 16 July,NaN
1,2,NaT,school_complaints,Siddharth Sawant,917506259177,sidd00787@gmail.com,Thane,NaN,Name of the child: Sudarshan Sawant\nAge:10\nG...,Ringing 08 July,Ringing 10 July,Ringing 11 July,Ringing 16 July,NaN,NaN
2,NaN,NaT,behavioral_issues,Swati Ambre,919372381762,swatiambre2@gmail.com,Thane,NaN,Ringing - 04 July,Ringing 10 July,Ringing 11 July,Ringing 16 July,Name of the child: Tanish Mahadik\nAge: 11\nLo...,Ringing 18 July,Ringing 21 July
3,NaN,NaT,learning_difficulties,Sheetal Kamble,18169606529,sheetalanandkamble@gmail.com,Mumbai,NaN,Repeated old enquiry which was received on 03 ...,Ringing 09 July,Ringing 10 July,NaN,NaN,NaN,NaN
4,NaN,2026-07-06,behavioral_issues,Mukesh Panthri,/,mukeshpanthri25@gmail.com,Pune,Name of the child Tejas\nAge:10\nLocation : Pu...,Ringing - 07 July,Ringing - 10 July,Ringing 11 July,NaN,NaN,NaN,NaN


In [5]:
print("Columns in dataset:")
print(df.columns.tolist())

print("\nDataset Info:")
df.info()

Columns in dataset:
['Sr. No ', 'Date', 'Concerns ', 'Name ', 'Contact number ', 'Email ID ', 'Location ', 'Remark 1 ', 'Remark 2 ', 'Remark 3 ', 'Consultation Done ', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14']

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 107 entries, 0 to 106
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Sr. No              15 non-null     object        
 1   Date                97 non-null     datetime64[ns]
 2   Concerns            103 non-null    object        
 3   Name                102 non-null    object        
 4   Contact number      103 non-null    object        
 5   Email ID            91 non-null     object        
 6   Location            101 non-null    object        
 7   Remark 1            100 non-null    object        
 8   Remark 2            73 non-null     object        
 9   Remark 3            40 non-null

In [6]:
# Standardize text across all columns for searching
df["Full_Text"] = df.astype(str).apply(lambda row: " ".join(row), axis=1)

# 1. Total Leads per Month
# Ensure Date column is parsed to extract month
if "Date" in df.columns:
    df["Date_Parsed"] = pd.to_datetime(
        df["Date"], errors="coerce", dayfirst=True
    )
    df["Month"] = df["Date_Parsed"].dt.strftime("%Y-%m")
    total_leads_per_month = df["Month"].value_counts().sort_index()
else:
    # Extract month pattern (e.g., -07-2026 or July) from text if no explicit Date column
    total_leads_per_month = df["Full_Text"].str.extract(
        r"(\d{2}-\d{2}-\d{2,4})", expand=False
    )

In [7]:
# 2. Ringing Customers
ringing_mask = df["Full_Text"].str.contains("Ringing", case=False, na=False)
ringing_customers = df[ringing_mask]

In [8]:
# 3. Interested Clients (Scheduled, Consultation Done, or Requested Callback)
interested_pattern = (
    r"consultation done|pgs|schedule|interested|will discuss|call back"
)
interested_mask = df["Full_Text"].str.contains(
    interested_pattern, case=False, na=False
)
interested_clients = df[interested_mask]

In [9]:
# 4. Potential Clients (Interested + Active Follow-ups)
potential_pattern = (
    r"will inform|schedule|discuss with husband|callback|ready for consultation"
)
potential_mask = df["Full_Text"].str.contains(
    potential_pattern, case=False, na=False
) & ~df["Full_Text"].str.contains(
    "not interested|cancelled|decline", case=False, na=False
)
potential_clients = df[potential_mask]

In [10]:
# 5. Interested but Not Converted (Showed interest/scheduled but declined/cancelled/didn't attend)
not_converted_after_interest_pattern = r"cancelled|declined|not picking|does not receive|not interested|cannot afford|distance issue"
interested_not_converted_mask = (
    interested_mask
    & df["Full_Text"].str.contains(
        not_converted_after_interest_pattern, case=False, na=False
    )
    & ~df["Full_Text"].str.contains(
        "consultation done", case=False, na=False
    )
)
interested_not_converted = df[interested_not_converted_mask]

In [11]:
# 6. Categorized Reasons for Not Converting
reasons_keywords = {
    "Distance / Location Issue": r"distance|location|far|cannot travel",
    "Not Comfortable with Online": r"online|not comfotable with online|hectic",
    "High Fees / Budget Constraint": r"cannot afford|fees|fee|budget",
    "Cancelled / Did Not Attend": r"cancelled|cancelled last moment|did not answer",
    "Taking Therapy Elsewhere / Local": r"in our city|other center|already taking|local",
    "Child Health / Exam Issues": r"exam|health issue|younger daughter",
    "No Concern / Misdial": r"does not have any concern|no concern|test lead",
}

reasons_summary = {}
for reason, pattern in reasons_keywords.items():
    reasons_summary[reason] = (
        df["Full_Text"].str.contains(pattern, case=False, na=False).sum()
    )

reasons_df = pd.DataFrame(
    list(reasons_summary.items()), columns=["Reason", "Count"]
).sort_values(by="Count", ascending=False)

In [12]:
# Display Summaries
print("=== 1. TOTAL LEADS PER MONTH ===")
print(total_leads_per_month, "\n")

print(f"=== 2. RINGING CUSTOMERS: {len(ringing_customers)} ===")
print(
    f"=== 3. INTERESTED CLIENTS: {len(interested_clients)} ==="
)
print(f"=== 4. POTENTIAL CLIENTS: {len(potential_clients)} ===")
print(
    f"=== 5. INTERESTED BUT NOT CONVERTED: {len(interested_not_converted)} ==="
)

print("\n=== 6. REASONS FOR NOT CONVERTING ===")
print(reasons_df.to_string(index=False))

=== 1. TOTAL LEADS PER MONTH ===
Month
2026-07    83
2026-08    14
Name: count, dtype: int64 

=== 2. RINGING CUSTOMERS: 56 ===
=== 3. INTERESTED CLIENTS: 33 ===
=== 4. POTENTIAL CLIENTS: 16 ===
=== 5. INTERESTED BUT NOT CONVERTED: 8 ===

=== 6. REASONS FOR NOT CONVERTING ===
                          Reason  Count
       Distance / Location Issue     59
     Not Comfortable with Online      7
            No Concern / Misdial      6
   High Fees / Budget Constraint      5
      Cancelled / Did Not Attend      4
      Child Health / Exam Issues      2
Taking Therapy Elsewhere / Local      1


In [14]:
with pd.ExcelWriter("Summary_Report.xlsx") as writer:
    df.to_excel(writer, sheet_name="All Leads", index=False)
    interested_clients.to_excel(
        writer, sheet_name="Interested Clients", index=False
    )
    potential_clients.to_excel(
        writer, sheet_name="Potential Clients", index=False
    )
    interested_not_converted.to_excel(
        writer, sheet_name="Interested Not Converted", index=False
    )
    ringing_customers.to_excel(
        writer, sheet_name="Ringing Customers", index=False
    )
    reasons_df.to_excel(
        writer, sheet_name="Drop Reasons Summary", index=False
    )

print("Report saved as 'Summary_Report.xlsx'")

Report saved as 'Summary_Report.xlsx'
